# Task 6 - Export DeBERTa Model & Tokenizer for Demo

Notebook này đóng gói checkpoint DeBERTa tốt nhất và tokenizer thành một thư mục sạch để Web UI hoặc script inference dùng lại. Notebook không train model mới.


In [ ]:
from pathlib import Path
import json
import shutil
import sys

import torch
from transformers import AutoTokenizer

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.modeling import DeBERTaDetector

MODEL_NAME = 'microsoft/deberta-v3-base'
TUNING_CONFIG_PATH = ROOT / 'outputs' / 'tv2_deberta_tuning' / 'best_config.json'
TUNED_CHECKPOINT_PATH = ROOT / 'outputs' / 'checkpoints' / 'deberta_tuned_best.pt'
CHECKPOINT_PATH = TUNED_CHECKPOINT_PATH

if TUNING_CONFIG_PATH.exists():
    best_cfg = json.loads(TUNING_CONFIG_PATH.read_text(encoding='utf-8'))
else:
    best_cfg = {}

EXPORT_DIR = ROOT / 'outputs' / 'final_deberta_demo'
TOKENIZER_DIR = EXPORT_DIR / 'tokenizer'
WEIGHTS_PATH = EXPORT_DIR / 'deberta_detector.pt'
CONFIG_PATH = EXPORT_DIR / 'model_config.json'
LABEL_PATH = EXPORT_DIR / 'label_mapping.json'

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

model_name_for_export = best_cfg.get('model_name', MODEL_NAME)
if isinstance(model_name_for_export, str) and model_name_for_export.startswith('/kaggle/') and not Path(model_name_for_export).exists():
    model_name_for_export = MODEL_NAME

runtime_config = {
    'architecture': 'DeBERTaDetector',
    'model_name': model_name_for_export,
    'weights_file': WEIGHTS_PATH.name,
    'tokenizer_dir': 'tokenizer',
    'max_length': int(best_cfg.get('max_length', 512)),
    'pooling': best_cfg.get('pooling', 'mean'),
    'dropout': float(best_cfg.get('dropout', 0.1)),
    'num_labels': int(best_cfg.get('num_labels', 2)),
    'probability_column': 'prob_ai',
    'source_checkpoint': str(CHECKPOINT_PATH),
    'source_tuning_config': str(TUNING_CONFIG_PATH) if TUNING_CONFIG_PATH.exists() else None,
}

print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Best config: {TUNING_CONFIG_PATH if TUNING_CONFIG_PATH.exists() else "not found, using defaults"}')
print(f'Export dir : {EXPORT_DIR}')
print(json.dumps(runtime_config, indent=2, ensure_ascii=False))


## 1. Check checkpoint


In [ ]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'Không tìm thấy checkpoint: {CHECKPOINT_PATH}. Hãy train DeBERTa trước khi export.'
    )

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
print('Checkpoint keys:', list(checkpoint.keys()) if isinstance(checkpoint, dict) else type(checkpoint))
if isinstance(checkpoint, dict) and 'best_auc' in checkpoint:
    print(f"Best validation AUC trong checkpoint: {checkpoint['best_auc']:.4f}")


## 2. Save tokenizer

Tokenizer cần được lưu cùng model để Web UI tokenize đúng cách, không phụ thuộc vào cache cục bộ.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.save_pretrained(TOKENIZER_DIR)
print(f'Saved tokenizer to: {TOKENIZER_DIR}')


## 3. Save weights and config for custom model

Model của đồ án dùng custom head `DeBERTaDetector`, nên ta lưu checkpoint `.pt` kèm config thay vì ép sang `AutoModelForSequenceClassification`.


In [ ]:
shutil.copy2(CHECKPOINT_PATH, WEIGHTS_PATH)

label_mapping = {
    '0': 'Human',
    '1': 'AI-generated',
}

CONFIG_PATH.write_text(json.dumps(runtime_config, indent=2, ensure_ascii=False), encoding='utf-8')
LABEL_PATH.write_text(json.dumps(label_mapping, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Saved weights      : {WEIGHTS_PATH}')
print(f'Saved model config : {CONFIG_PATH}')
print(f'Saved label mapping: {LABEL_PATH}')


## 4. Smoke test load model

Cell này chỉ kiểm tra checkpoint load được vào kiến trúc hiện tại. Inference thật cho Web UI có thể dùng `src/inference.py`.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeBERTaDetector(
    model_name=runtime_config['model_name'],
    num_labels=2,
    dropout=runtime_config['dropout'],
    pooling=runtime_config['pooling'],
).to(device)

ckpt = torch.load(WEIGHTS_PATH, map_location=device, weights_only=False)
state_dict = ckpt.get('state_dict', ckpt)
model.load_state_dict(state_dict)
model.eval()

print(f'Model loaded successfully on {device}.')


## 5. Export summary


In [ ]:
readme = f"""# Final DeBERTa Demo Export

Files:
- `{WEIGHTS_PATH.name}`: checkpoint for `src.modeling.DeBERTaDetector`
- `tokenizer/`: Hugging Face tokenizer files
- `{CONFIG_PATH.name}`: model/runtime config
- `{LABEL_PATH.name}`: label mapping

Recommended inference path:
```bash
python -m src.inference --input input.csv --checkpoint {WEIGHTS_PATH} --output predictions.csv --text-col text_clean
```
"""
(EXPORT_DIR / 'README_EXPORT.md').write_text(readme, encoding='utf-8')
print(readme)
